# Istanbul Healthcare Accessibility Analysis
## Part 1: Data Exploration

**Author:** Arife Mutlu  
**Date:** January 22, 2026  
**Goal:** Explore healthcare facility distribution in Istanbul

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src') if 'notebooks' not in os.getcwd() else os.path.join(os.getcwd(), '..', 'src'))

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium

from load_data import load_sample_facilities, load_istanbul_districts, load_facilities, make_geodataframe
from buffer_analysis import create_buffers, coverage_percentage
from find_nearest import find_nearest

pd.set_option('display.max_columns', None)
%matplotlib inline

## 1. Load Sample Data

In [ ]:
facilities = load_facilities()
facilities_gdf = make_geodataframe(facilities)
print(f"Loaded {len(facilities_gdf)} facilities")
facilities_gdf.head(10)

## 2. Basic Exploration

In [ ]:
print(f"CRS: {facilities_gdf.crs}")
print(f"\nFacility types:")
print(facilities_gdf['type'].value_counts())
print(f"\nDistricts covered: {facilities_gdf['district'].nunique()}")
print(facilities_gdf['district'].value_counts())

In [ ]:
type_counts = facilities_gdf['type'].value_counts()
type_counts.plot(kind='bar', color=['red', 'blue', 'green'], figsize=(8, 5))
plt.title('Healthcare Facility Distribution in Istanbul')
plt.xlabel('Facility Type')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../outputs/facility_distribution.png', dpi=150)
plt.show()

## 3. Interactive Map

In [ ]:
color_map = {'hastane': 'red', 'sağlık_merkezi': 'blue', 'sağlık_ocağı': 'green'}

m = folium.Map(location=[41.0082, 28.9784], zoom_start=11, tiles='OpenStreetMap')

for idx, row in facilities_gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=8,
        popup=f"<b>{row['name']}</b><br>Tip: {row['type']}<br>İlçe: {row['district']}",
        color=color_map.get(row['type'], 'gray'),
        fill=True,
        fillOpacity=0.7
    ).add_to(m)

m.save('../outputs/facility_map.html')
m

## 4. Buffer Analysis

In [ ]:
districts = load_istanbul_districts()
buffers = create_buffers(facilities_gdf, distances_km=[2, 5, 10])

for km, buf_gdf in buffers.items():
    pct = coverage_percentage(districts, buf_gdf)
    print(f"{km}km buffer → {pct:.1f}% of Istanbul area covered")

In [ ]:
# Visualise 5km buffer on map
m2 = folium.Map(location=[41.0082, 28.9784], zoom_start=10)

buf_5km = buffers[5]
folium.GeoJson(
    buf_5km.__geo_interface__,
    style_function=lambda f: {'fillColor': 'blue', 'color': 'blue', 'fillOpacity': 0.2}
).add_to(m2)

for idx, row in facilities_gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color='red', fill=True, fillOpacity=0.8,
        popup=row['name']
    ).add_to(m2)

m2.save('../outputs/buffer_5km_map.html')
m2

## 5. Nearest Facility Analysis

In [ ]:
taksim = (28.9850, 41.0369)
nearest = find_nearest(taksim, facilities_gdf, k=3)

print("3 nearest facilities to Taksim Square:")
nearest['distance_km'] = (nearest['distance_m'] / 1000).round(2)
print(nearest[['name', 'type', 'distance_km']])

## Next Steps

- [ ] Download real Istanbul district boundaries from OSM
- [ ] Collect actual healthcare facility data from Istanbul Municipality open data
- [ ] Perform district-level density analysis (facilities per capita)
- [ ] Identify underserved areas (low coverage + high population)